# Speech rate per sentence, and outliers

Reads the **player HTML files** exported by the condition notebooks (`…_block1.html`, …, or any player made with `audiovisualize_interactive`). They already contain every token with its start and end time, so nothing is synthesized and no model is loaded.

For every sentence (BOS … EOS):

| measure | definition |
|---|---|
| `speech_rate` | phonemes / s over the sentence (lead / trail silence excluded, pauses inside included) |
| `artic_rate`  | phonemes / s over the phonemes only (with their diacritics and stress marks; spaces and punctuation excluded) |
| `syll_rate`   | vowels (≈ syllables) / s over the sentence |
| `wpm`         | words / minute over the sentence |
| `pause_s`     | time in spaces and punctuation inside the sentence |

**Outliers.** TTS speaks long sentences faster (or slower) than short ones, so a sentence is only unusual *for its length*: within every file, the rate is regressed on log(number of phonemes), and sentences whose residual is more than `Z` robust standard deviations away (median / MAD) are flagged as **fast** or **slow**.

**Continuous rate.** Every phoneme is a point at its centre time, smoothed with a Gaussian of `SIGMA` seconds → phonemes per second at every moment (pauses show as dips).

Functions are in `rate_tools.py` (next to this notebook); `help(flag_outliers)` etc. show the details.

In [ ]:
from rate_tools import *
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

## Settings

In [ ]:
FILES    = "output_online/consistent/*_block*.html"   # a glob pattern or a list of files
#          "output_online/*/*_block*.html"            #  -> every condition at once
#          (the "_full.html" files contain the same sentences again, leave them out to avoid counting twice)
MEASURE  = "artic_rate"     # artic_rate | speech_rate | syll_rate | wpm   (used for the outliers)
Z        = 2.5              # robust z beyond which a sentence is an outlier
GROUP_BY = "file"           # outliers judged within each "file", "condition", or None = all together
SIGMA    = 0.5              # smoothing of the continuous rate, in seconds
OUT_CSV  = "output_online/speech_rate_sentences.csv"

## Load the files and measure every sentence

In [ ]:
players = load_players(FILES)
rows = sentence_table(players)
outliers = flag_outliers(rows, measure=MEASURE, z=Z, group_by=GROUP_BY)

## Summary per file

In [ ]:
UNIT = {"artic_rate": "phon/s", "speech_rate": "phon/s", "syll_rate": "syll/s", "wpm": "words/min"}[MEASURE]
print(f"{'file':<38}{'sent':>5}{'median':>9}{'IQR':>15}{'min':>7}{'max':>7}{'slope':>8}{'fast':>6}{'slow':>6}   ({MEASURE}, {UNIT})")
for p in players:
    rs = [r for r in rows if r["file"] == p["name"]]
    v = np.array([r[MEASURE] for r in rs])
    q1, q3 = np.percentile(v, [25, 75])
    print(f"{p['name']:<38}{len(rs):>5}{np.median(v):>9.2f}{f'{q1:.2f}–{q3:.2f}':>15}{v.min():>7.2f}{v.max():>7.2f}"
          f"{rs[0]['slope_per_log_length']:>8.2f}{sum(r['outlier'] == 'fast' for r in rs):>6}{sum(r['outlier'] == 'slow' for r in rs):>6}")
print("\nslope = change in rate per unit of log(phonemes): > 0 long sentences are faster, < 0 slower")

## The outlier sentences

In [ ]:
if not outliers:
    print("no outliers")
for r in sorted(outliers, key=lambda r: -abs(r["z"])):
    print(f"{r['outlier']:>4}  z={r['z']:+5.1f}  {r['file']}  sentence {r['sentence']:>3}  ({r['start']:.1f}–{r['end']:.1f} s)"
          f"  {r['n_phones']} phonemes  {r[MEASURE]:.2f} vs expected {r['fit']:.2f} {UNIT}"
          f"  pause {r['pause_s']:.2f} s\n        {r['text'][:140]}")

## Rate against sentence length
One panel per file: every dot is a sentence, the line is the expected rate for that length; outliers are labelled with their sentence number.

In [ ]:
BLUE, FAST, SLOW, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#6250d6", "#0b0b0b", "#52514e", "#e7e6e2"
def style(ax):
    ax.spines[["top", "right"]].set_visible(False); ax.spines[["left", "bottom"]].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0); ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)

n = len(players); cols = min(3, n); nr = int(np.ceil(n / cols))
fig, axes = plt.subplots(nr, cols, figsize=(4.4 * cols, 3.4 * nr), squeeze=False, sharey=True)
for ax, p in zip(axes.flat, players):
    rs = [r for r in rows if r["file"] == p["name"]]
    x = np.array([r["n_phones"] for r in rs]); y = np.array([r[MEASURE] for r in rs])
    col = [FAST if r["outlier"] == "fast" else SLOW if r["outlier"] == "slow" else BLUE for r in rs]
    ax.scatter(x, y, s=34, c=col, edgecolor="white", linewidth=1, zorder=3)
    order = np.argsort(x)
    ax.plot(x[order], np.array([r["fit"] for r in rs])[order], color=INK, lw=1.2, ls="--")
    for r in rs:
        if r["outlier"]:
            ax.annotate(str(r["sentence"]), (r["n_phones"], r[MEASURE]), xytext=(4, 3), textcoords="offset points",
                        fontsize=8, color=INK)
    ax.set_title(p["name"], loc="left", fontsize=9, color=INK); style(ax)
    ax.set_xlabel("phonemes in the sentence", color=MUTED)
for ax in axes[:, 0]:
    ax.set_ylabel(f"{MEASURE} ({UNIT})", color=MUTED)
for ax in list(axes.flat)[n:]:
    ax.axis("off")
handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=l) for c, l in ((BLUE, "sentence"), (FAST, "fast outlier"), (SLOW, "slow outlier"))]
handles.append(plt.Line2D([], [], color=INK, ls="--", label="expected for the length"))
fig.legend(handles=handles, loc="upper right", ncol=4, frameon=False, fontsize=8)
fig.suptitle("Rate against sentence length", x=0.01, ha="left", fontsize=11)
plt.tight_layout(rect=(0, 0, 1, 0.95)); plt.show()

## Continuous rate through time
Phonemes per second at every moment (Gaussian smoothing of `SIGMA` s). The flat grey line is each sentence's own `speech_rate`; outlier sentences are shaded.

In [ ]:
fig, axes = plt.subplots(len(players), 1, figsize=(12, 2.4 * len(players) + 0.4), squeeze=False)
for ax, p in zip(axes[:, 0], players):
    t, rate = rate_curve(p, sigma=SIGMA)
    rs = [r for r in rows if r["file"] == p["name"]]
    for r in rs:
        if r["outlier"]:
            ax.axvspan(r["start"], r["end"], color=FAST if r["outlier"] == "fast" else SLOW, alpha=0.15, lw=0)
        ax.hlines(r["speech_rate"], r["start"], r["end"], color="#9ca3af", lw=2)
        ax.axvline(r["start"], color=GRID, lw=0.8)
    ax.plot(t, rate, color=BLUE, lw=1.4)
    ax.set_title(p["name"], loc="left", fontsize=9, color=INK); style(ax)
    ax.set_ylabel("phon/s", color=MUTED); ax.set_xlim(0, p["duration"])
axes[-1, 0].set_xlabel("time (s)", color=MUTED)
fig.suptitle("Continuous speech rate", x=0.01, ha="left", fontsize=11)
plt.tight_layout(rect=(0, 0, 1, 1 - 0.35 / fig.get_figheight())); plt.show()

## Rate per sentence, in story order

In [ ]:
fig, axes = plt.subplots(len(players), 1, figsize=(12, 2.0 * len(players) + 0.4), squeeze=False, sharey=True)
for ax, p in zip(axes[:, 0], players):
    rs = [r for r in rows if r["file"] == p["name"]]
    x = [r["sentence"] for r in rs]
    ax.bar(x, [r[MEASURE] for r in rs], 0.75,
           color=[FAST if r["outlier"] == "fast" else SLOW if r["outlier"] == "slow" else BLUE for r in rs])
    ax.plot(x, [r["fit"] for r in rs], color=INK, lw=1, ls="--")
    ax.set_title(p["name"], loc="left", fontsize=9, color=INK); style(ax)
    ax.set_ylabel(UNIT, color=MUTED); ax.xaxis.set_major_locator(MaxNLocator(integer=True))
axes[-1, 0].set_xlabel("sentence", color=MUTED)
fig.suptitle(f"{MEASURE} per sentence (dashed = expected for its length)", x=0.01, ha="left", fontsize=11)
plt.tight_layout(rect=(0, 0, 1, 1 - 0.35 / fig.get_figheight())); plt.show()

## Files compared

In [ ]:
fig, ax = plt.subplots(figsize=(max(5, 1.3 * len(players) + 2), 3.6))
data = [[r[MEASURE] for r in rows if r["file"] == p["name"]] for p in players]
ax.boxplot(data, widths=0.5, showfliers=False, medianprops=dict(color=INK),
           boxprops=dict(color=MUTED), whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED))
for i, (p, d) in enumerate(zip(players, data), start=1):
    rs = [r for r in rows if r["file"] == p["name"]]
    jitter = np.random.default_rng(0).uniform(-0.12, 0.12, len(d))
    ax.scatter(i + jitter, d, s=16, zorder=3,
               c=[FAST if r["outlier"] == "fast" else SLOW if r["outlier"] == "slow" else BLUE for r in rs])
ax.set_xticks(range(1, len(players) + 1), [p["name"] for p in players], rotation=20, ha="right", fontsize=8)
ax.set_ylabel(f"{MEASURE} ({UNIT})", color=MUTED); style(ax)
ax.set_title("Sentence rates per file", loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## Save

In [ ]:
save_csv(rows, OUT_CSV)